# XCalib quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/radar-lab/XCalib/blob/main/demo/xcalib_quickstart.ipynb)

Camera–LiDAR cross-modal matching for edge devices. This notebook goes from
`pip install` to a calibrated **LiDAR → camera** projection on a real A9
(TUMTraf) intersection frame, using the released `crlite` weights.

- Repo: <https://github.com/radar-lab/XCalib> · Docs: <https://radar-lab.github.io/XCalib/>
- **Hub-first, with a local fallback:** the cells pull released weights and the
  A9 test cache from Hugging Face; when those aren't reachable (the dataset is
  licence-gated) they fall back to the committed sample, and to a local
  checkpoint in a source checkout — so a checkout runs end-to-end offline.

## What you'll do

1. Install XCalib and load the `crlite` matcher (**Hugging Face weights**).
2. Pull one A9 test frame (**Hugging Face dataset**, with an offline fallback).
3. Score every camera × LiDAR detection pair with `matcher.match()` and view the overlay.
4. Recover the camera–LiDAR extrinsics with `matcher.oneshot()` + `calibrate()` and
   view the LiDAR projection.

Inputs are always four plain arrays — `image`, `point_cloud`, `bboxes_2d`,
`bboxes_3d` — so this drops into any perception pipeline
([input protocol](https://radar-lab.github.io/XCalib/protocol/)).

In [ ]:
# 1a. Install the package only if it isn't already importable (no-op in the demo env).
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("xcalib") is None:
    # Until XCalib is published to PyPI, swap this for:
    #   git+https://github.com/radar-lab/XCalib
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "xcalib[train]"], check=True)

In [ ]:
# 1b. Make sure we're somewhere with the committed sample + demo helpers.
# In a local `demo/` checkout this is a no-op; on Colab it shallow-clones the repo.
import os
import subprocess
import sys

if not os.path.isdir("frames/a9_sample"):
    if os.path.isdir("demo/frames/a9_sample"):
        os.chdir("demo")
    else:
        subprocess.run(
            ["git", "clone", "-q", "--depth", "1",
             "https://github.com/radar-lab/XCalib", "_xcalib_src"], check=False)
        if os.path.isdir("_xcalib_src/demo"):
            os.chdir("_xcalib_src/demo")
sys.path.insert(0, os.getcwd())
print("working dir:", os.getcwd())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import xcalib
from xcalib import Matcher
from xcalib.visualization import draw_matching_overlay, draw_calibration_overlay

print("xcalib", xcalib.__version__)

def show(rgb, title=""):
    plt.figure(figsize=(11, 5))
    plt.imshow(rgb)
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()

## 1 · Load the matcher

`Matcher.from_pretrained` pulls the released `crlite` weights for a site from the
Hugging Face Hub. If a local checkpoint is present (a source checkout), we use it
to stay offline — pass `weights=None` to force the Hub path.

In [ ]:
from pathlib import Path

SITE = "a9_dataset_r02_s01"
candidates = [
    Path("weights") / f"crlite_{SITE}_best.pth",
    Path("../checkpoints") / f"crlite_{SITE}_best.pth",
]
local_w = next((p for p in candidates if p.exists()), None)

matcher = Matcher.from_pretrained("crlite", site=SITE, weights=local_w, device="auto")
print("loaded crlite |", "local weights" if local_w else "Hub weights",
      "| device:", matcher.device)

## 2 · Get a frame

The real path streams the released A9 test cache from the Hub via
`xcalib.load_dataset`. **An A9 cache stores detections from *both* s110 cameras in
one frame**, so `UTCFrame.for_camera(...)` keeps just one camera's 2D boxes (paired
with that camera's image) — otherwise the image and its boxes wouldn't share a
pixel coordinate system and you'd be matching two cameras at once. If the Hub isn't
reachable (or the dataset licence gate is still pending), we fall back to the
committed sample (already single-camera). Either way we get the same four arrays
for one camera.

In [ ]:
from demo_config import SAMPLE_FRAMES

# One camera, used for both the Hub path and the committed-sample fallback.
CAMERA = "s110_camera_basler_south1_8mm"
dataset_gt = {}   # filled on the Hub path: released K + scene extrinsics (ground truth)


def demo_frames(limit=16):
    # Hub path: stream the released A9 test cache, one camera per frame.
    try:
        from xcalib import load_dataset
        ref_E, yielded = None, 0
        with load_dataset(SITE, split="test") as loader:
            for raw in loader:
                if CAMERA not in raw.images:
                    continue
                # The A9 test split concatenates scenes with DIFFERENT extrinsics;
                # accumulate one scene so the single calibration stays consistent
                # (and can be checked against that scene's ground-truth pose).
                E = raw.extrinsics.get(CAMERA)
                if E is not None:
                    if ref_E is None:
                        ref_E = E
                        dataset_gt.update(extrinsics=E, intrinsics=raw.intrinsics.get(CAMERA))
                    elif not np.allclose(E, ref_E, atol=1e-3):
                        continue
                # for_camera returns (image, pc, bboxes_2d, bboxes_3d) for CAMERA.
                yield raw.for_camera(CAMERA)
                yielded += 1
                if yielded >= limit:
                    return
        return
    except Exception as exc:
        print("Hub dataset unavailable, using committed sample:", type(exc).__name__)

    # Offline fallback: the committed sample (already single-camera, single-scene).
    from demo_common import iter_demo_frames
    for f in iter_demo_frames(sample_frames=SAMPLE_FRAMES, site=SITE, split="test",
                              camera_name=CAMERA, limit=limit):
        yield f.image, f.point_cloud, f.bboxes_2d, f.bboxes_3d


frames = list(demo_frames(16))
image, point_cloud, bboxes_2d, bboxes_3d = frames[0]
print(f"{len(frames)} frames | image {image.shape} | points {point_cloud.shape} |",
      f"{len(bboxes_2d)} camera boxes | {len(bboxes_3d)} LiDAR boxes")
show(image, "input frame (one camera)")

## 3 · Match camera ↔ LiDAR

`matcher.match()` scores every camera × LiDAR detection pair and keeps each camera
box's best LiDAR match above the threshold. Green links span the image and the
bird's-eye LiDAR view; below-threshold detections stay unmatched on purpose.

In [ ]:
result = matcher.match(image, point_cloud, bboxes_2d, bboxes_3d,
                       top_k=1, match_threshold=0.5)

print("matches (camera_det -> lidar_det, score):")
for i, j, s in result.matches:
    print(f"  {i:>2} -> {j:>2}   {s:.3f}")

show(draw_matching_overlay(image, point_cloud, bboxes_2d, bboxes_3d,
                           result.matches, match_threshold=0.5),
     "matching: camera detections -> LiDAR detections")

## 4 · Calibrate (targetless extrinsics)

Those matches are the 2D–3D correspondences `matcher.oneshot()` feeds to PnP/RANSAC
to recover the camera–LiDAR extrinsics — no targets.

A roadside scene is nearly **planar** (every box centre sits on the road), so a
single frame underconstrains PnP and admits a degenerate "clump" pose. We stream a
few frames; `calibrate()` re-solves the whole buffer and (since 0.2) scores each
solve over *all* correspondences, **rejecting the clump** (`result.accepted is
False`) and adopting the latest well-conditioned pose as objects sweep the scene.

The intrinsics come from the dataset itself: `load_dataset` exposes the A9 cache's
`/calibration` as `frame.intrinsics[camera]` (the released `K`), with the bundled
south1 file as the offline fallback.

In [ ]:
from xcalib import CameraIntrinsics


def camera_intrinsics(camera):
    # The released K is captured from the dataset's /calibration while streaming
    # frames above (frame.intrinsics[camera]); offline we use the bundled file.
    if dataset_gt.get("intrinsics") is not None:
        return CameraIntrinsics.from_matrix(dataset_gt["intrinsics"])
    from demo_common import load_intrinsics
    from demo_config import INTRINSICS_JSON
    return load_intrinsics(INTRINSICS_JSON)


K = camera_intrinsics(CAMERA)
print(f"intrinsics: fx={K.fx:.1f} fy={K.fy:.1f} cx={K.cx:.1f} cy={K.cy:.1f}")
session = matcher.oneshot(K, match_threshold=0.5)

final = None
for img, pc, b2, b3 in frames:
    session.observe(img, pc, b2, b3)
    cal = session.calibrate(min_pairs=12)
    if cal.success and cal.accepted:
        final = (cal, img, pc, b3)
        print(f"accepted : {cal.n_inliers:>3}/{cal.n_correspondences} inliers, "
              f"buffer-median {cal.buffer_reproj_px:5.1f}px")
    elif cal.success:
        print(f"rejected : degenerate planar pose "
              f"(buffer-median {cal.buffer_reproj_px:5.1f}px)")

cal, img, pc, b3 = final
print(f"\nfinal calibration over {cal.n_correspondences} pairs, "
      f"buffer-median {cal.buffer_reproj_px:.1f}px")
show(draw_calibration_overlay(img, pc, projection=cal.projection, bboxes_3d=b3),
     "calibration: LiDAR projected onto the camera")

## 5 · Validate against ground truth (optional)

The A9 cache also ships the camera-LiDAR **extrinsics** (`frame.extrinsics[camera]`,
a 4x4 lidar->camera pose). `result.pose_error(...)` reports how far the targetless
estimate is from that released pose — the rotation/translation a paper-style table
quotes — and we render the ground-truth projection for comparison. The committed
sample carries no calibration, so this runs only on the Hub path.

In [ ]:
if dataset_gt.get("extrinsics") is not None:
    E = np.asarray(dataset_gt["extrinsics"])
    rot_deg, trans_m = cal.pose_error(E)
    print(f"targetless vs ground truth: rotation {rot_deg:.2f} deg, "
          f"translation {trans_m:.2f} m  ({cal.n_correspondences} pairs)")
    P_gt = np.asarray(dataset_gt["intrinsics"]) @ E[:3, :4]
    show(draw_calibration_overlay(img, pc, projection=P_gt, bboxes_3d=b3),
         "ground-truth calibration (released K + extrinsics)")
else:
    print("Ground-truth calibration lives in the A9 Hub cache, not the committed sample.")

## Where to go next

- **Full edge loop:** `demo/run_03_full_demo.py` runs this match → refine-calibrate
  loop as a streaming script (the demo this notebook is based on).
- **One-shot adaptation:** after calibrating, `session.observe(...)` banks
  geometry-confirmed pairs and `session.adapt(steps=...)` fine-tunes the matcher
  on-site, label-free.
- **Export:** `matcher.build("onnx", ...)` / `matcher.build("trt", precision="fp16")`
  for Jetson deployment.
- **Docs:** [input protocol](https://radar-lab.github.io/XCalib/protocol/) ·
  [models & datasets](https://radar-lab.github.io/XCalib/hub/) ·
  [reproducing the paper](https://radar-lab.github.io/XCalib/paper/)